### Introduction to Tier 1 Disease Prediction

This notebook focuses on upgrading the AyurFit project from an unsupervised semantic search model (Cosine Similarity) to a structured Two-Tier Machine Learning Architecture. Specifically, it builds the Tier 1 Disease Prediction Model using dense vector embeddings and a Support Vector Machine classifier.

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
import joblib

### Explanation of data loading

An essential first step in the machine learning pipeline is loading the dataset. We will ingest the patient data from our CSV file into a structured Pandas DataFrame to facilitate exploratory analysis and feature engineering.

In [ ]:
df = pd.read_csv('../dataset/final ayurfit.csv')
df.head()

### Task 1: Fix the Data Imbalance (Data Cleaning Cell)

Before label encoding, we count the frequency of each Disease. We filter the DataFrame to only keep rows where the Disease appears at least 2 times. This prevents the `UndefinedMetricWarning` during the train/test split and ensures we have enough data for a balanced split.

In [ ]:
df = df.dropna(subset=['Symptoms', 'Disease'])
disease_counts = df['Disease'].value_counts()
df = df[df['Disease'].isin(disease_counts[disease_counts >= 2].index)]

### Explanation of Target Encoding

Machine learning classifiers require numerical targets. We utilize `LabelEncoder` to convert the categorical `Disease` labels into a numerical format, establishing a clear mapping that can be reversed during inference.

In [ ]:
label_encoder = LabelEncoder()
df['Disease_Encoded'] = label_encoder.fit_transform(df['Disease'])

### Explanation of utilizing GPU for NLP Feature Extraction

We transition from traditional bag-of-words approaches to state-of-the-art transformer models. We initialize the `all-MiniLM-L6-v2` SentenceTransformer, explicitly utilizing the GPU to significantly accelerate the generation of semantic embeddings.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

### Explanation of generating dense vector embeddings

We transform the raw textual `Symptoms` into a rich mathematical representation. The transformer encodes these texts into 384-dimensional dense vectors, capturing semantic meaning and nuance, which will serve as our primary feature matrix (X).

In [ ]:
X = model.encode(df['Symptoms'].tolist(), show_progress_bar=True)
y = df['Disease_Encoded'].values

### Task 2: Update Train/Test Split

We update the `train_test_split` function to include `stratify=y`. This ensures every disease is proportionally represented in both the training and testing sets. We use `test_size=0.3` to ensure the mathematical conditions for stratifying hundreds of classes are met.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Support Vector Machine Training

Support Vector Machines (SVM), specifically `LinearSVC`, are highly effective in high-dimensional spaces, making them an excellent choice for 384-dimensional text embeddings. The model learns optimal hyperplanes to separate the distinct disease classes based on their symptom embeddings.

In [ ]:
svm_classifier = LinearSVC(random_state=42, dual=False)
svm_classifier.fit(X_train, y_train)

### Task 3: Print Overall Accuracy

We generate the `classification_report` on the present classes to eliminate 0-support noise. Afterwards, we specifically calculate and print the overall `accuracy_score` as a clean percentage.

In [ ]:
y_pred = svm_classifier.predict(X_test)
present_classes = np.unique(np.concatenate((y_test, y_pred)))
target_names_present = label_encoder.inverse_transform(present_classes)
print(classification_report(y_test, y_pred, labels=present_classes, target_names=target_names_present, zero_division=0))

overall_accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {overall_accuracy * 100:.1f}%")

### Task 4: Create a Manual Testing Function

We define a function called `predict_custom_symptoms(symptoms_text)` to test out the model dynamically on new strings. It takes string input, uses the transformer to encode it into a 384-dimensional array, passes it to the SVM for prediction, and applies inverse-transform to extract the human-readable disease name. We also calculate the prediction confidence by applying a softmax over the model's decision function logits.

In [ ]:
def predict_custom_symptoms(symptoms_text):
    # 1. Generate the 384-dimensional embedding for the input string
    embedding = model.encode([symptoms_text], show_progress_bar=False)
    
    # 2. Predict the encoded label using the SVM model
    predicted_encoded = svm_classifier.predict(embedding)
    
    # 3. Calculate a confidence percentage using the decision function (Softmax)
    decision_scores = svm_classifier.decision_function(embedding)
    exp_scores = np.exp(decision_scores - np.max(decision_scores)) # Prevent overflow
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    confidence = np.max(probabilities) * 100
    
    # 4. Inverse-transform the label back into the readable Disease string
    predicted_disease = label_encoder.inverse_transform(predicted_encoded)[0]
    
    # 5. Print the input symptoms, predicted disease, and confidence
    print(f"Input Symptoms: '{symptoms_text}'")
    print(f"Predicted Disease: {predicted_disease} (Confidence: {confidence:.1f}%)\n")
    
    return predicted_disease, confidence

### Task 5: Run the Tests

We call `predict_custom_symptoms()` with 3 distinct manual test cases based on typical dataset symptoms to verify that our NLP + SVM pipeline correctly classifies general inputs.

In [ ]:
predict_custom_symptoms('severe chest pain and shortness of breath')
predict_custom_symptoms('frequent urination and extreme thirst')
predict_custom_symptoms('itchy red skin rash')

### Task 6: Export Models

Finally, we serialize our updated SVM classifier and label encoder using `joblib`. Exporting these artifacts enables seamless integration into the production backend.

In [ ]:
joblib.dump(svm_classifier, 'tier1_svm_model.joblib')
joblib.dump(label_encoder, 'disease_label_encoder.joblib')